## Creates new transactions table 
## has two additional columns: elus and koht (with values YES/UNK)

In [1]:
import sqlite3
import pandas as pd

## Configuration

In [2]:
# transaktsioonide andmebaas
TRANSACTION_DB = "../example_data/transactions.db"

# loodud tabelid
ENRICHED_TRANSACTIONS = "../example_data/enriched_transactions.db"

# määruste tabelid
PHRASE_PATTERNS = "../example_data/phrase_patterns.db"

# uus transaktsioonide tabel
transactions2 = "transaction_v2"

# tabelite nimed elusolendite ja kohtade jaoks
elustabel = 'elus_v1'
kohttabel = 'koht_v1'

## Connect to db

In [3]:
con = sqlite3.connect(ENRICHED_TRANSACTIONS)
cur = con.cursor()

# ainult esmakordseks db faili loomise ajaks
#cur.execute('pragma encoding=UTF8') 

# transaktsioonide andmebaasi lisamine
cur.execute(f'ATTACH DATABASE "{TRANSACTION_DB}" AS trans')

# määruste andmebaasi lisamine (maarused)
cur.execute(f'ATTACH DATABASE "{PHRASE_PATTERNS}" AS maarused')

## Workflow

## Create new transaction table

### juurde lisada veerud koht ja elus, kus on listide põhjal otsus sõna kohta

In [4]:
%%time
cur.execute("""DROP TABLE if exists {new_table}""".format(new_table = transactions2))
cur.execute("""CREATE TABLE {new_table} as SELECT * FROM trans.'transaction' """.format(new_table = transactions2))

CPU times: user 4.42 ms, sys: 1.07 ms, total: 5.49 ms
Wall time: 9.76 ms


In [5]:
# add default values for koht and elus
cur.execute("""
ALTER TABLE {new_table}
ADD koht VARCHAR(50) DEFAULT 'UNK'
""".format(new_table = transactions2))
con.commit()

cur.execute("""
ALTER TABLE {new_table}
ADD elus VARCHAR(50) DEFAULT 'UNK'
""".format(new_table = transactions2))
con.commit()

#### fill in new columns

In [6]:
%%time

cur.execute("""
UPDATE {new_table}
SET elus = 'YES'
WHERE lower({new_table2}.lemma) in (select lower(lemma) from maarused.{elustbl})
""".format(new_table = transactions2, new_table2 = transactions2, elustbl=elustabel))

con.commit()

CPU times: user 6.8 ms, sys: 2.04 ms, total: 8.84 ms
Wall time: 13.7 ms


In [7]:
%%time

cur.execute("""
UPDATE {new_table}
SET koht = 'YES'
WHERE lower({new_table2}.lemma) in (select lower(lemma) from maarused.{kohttbl})
""".format(new_table = transactions2, new_table2 = transactions2, kohttbl=kohttabel))

con.commit()

CPU times: user 3.87 ms, sys: 1.41 ms, total: 5.28 ms
Wall time: 7.84 ms


## kontroll

In [9]:
query = """SELECT * from {new_table} 
where elus='YES'
limit 5""".format(new_table = transactions2)
source = pd.read_sql_query(query, con)
source

,id,head_id,loc,loc_rel,deprel,form,lemma,feats,parent_loc,pos,koht,elus
0,4,3,1,-3,obj,Bändi,bänd,"adit,com,sg",None,S,UNK,YES
1,9,4,4,-2,nsubj,solist,solist,"com,nom,sg",None,S,UNK,YES
2,19,10,4,1,obl,sul,sina,"ad,sg",None,P,UNK,YES
3,21,11,1,-1,nsubj,Ma,mina,"nom,sg",None,P,UNK,YES
4,23,11,6,2,obl,juhul,juht,"ad,com,sg",None,S,UNK,YES


In [10]:
con.close()